# Stemmekloning i Google Colab — IndexTTS2 Premium (SECourses)

Denne notatblokka installerer **SECourses sitt grensesnitt** til IndexTTS2, ikke det enkle
standardgrensesnittet. Det gir dere forhåndsinnstillinger som kan lagres, opptak rett fra
mikrofonen, avbryt-knapp, og følelsesvalg på engelsk.

**Les dette først:**

1. Klikk **Kjøretid → Endre type kjøretid → Maskinvareakselerator: T4 GPU → Lagre.**
   Uten GPU kommer ingenting til å fungere.
2. Kjør cellene i rekkefølge, ovenfra og ned.
3. Steg 1 og 2 tar til sammen **15–25 minutter**. Start dem med én gang, og gjør opptakene
   mens de kjører.
4. Alt forsvinner når Colab-økta lukkes. **Last ned lydfilene før dere går** (Steg 5).

> Dere kloner bare deres egen stemme. Ingenting publiseres.


## Steg 0 – Sjekk at dere har fått GPU

In [ ]:
!nvidia-smi

import torch
print()
print("GPU tilgjengelig:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Skjermkort:", torch.cuda.get_device_name(0))
else:
    print("STOPP: Kjoretid -> Endre type kjoretid -> T4 GPU, og kjor denne cellen paa nytt.")


## Steg 1 – Hent appen og installer

Vi henter SECourses-appen fra GitHub og bygger et eget Python 3.10-miljø til den.
Colab kjører en nyere Python enn appen tåler, så `uv` laster ned 3.10 for oss.

**Tar 8–15 minutter.** Mange linjer med tekst er normalt.

In [1]:
%cd /content
!git clone https://github.com/FurkanGozukara/Premium_IndexTTS2_SECourses
%cd /content/Premium_IndexTTS2_SECourses
!git pull

!pip install -q -U uv
print("\nuv installert.")


/content
Cloning into 'Premium_IndexTTS2_SECourses'...
remote: Enumerating objects: 393, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 393 (delta 15), reused 13 (delta 4), pack-reused 356 (from 1)
Receiving objects: 100% (393/393), 31.55 MiB | 19.51 MiB/s, done.
Resolving deltas: 100% (112/112), done.
Filtering content: 100% (15/15), 12.53 MiB | 6.32 MiB/s, done.
/content/Premium_IndexTTS2_SECourses
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 97.1 MB/s eta 0:00:00

uv installert.


Appen leveres normalt med en `requirements.txt` i zip-fila fra Patreon. Den ligger ikke i
GitHub-repoet, så vi skriver en Colab-tilpasset versjon her.

**To pakker er med vilje fjernet:** `flash_attn` og `sageattention`. De gjør appen raskere,
men krever et nyere skjermkort enn T4-en Colab gir bort gratis. Prøver man å bruke dem på en
T4, krasjer appen.

In [2]:
requirements = """--extra-index-url https://download.pytorch.org/whl/cu129
torch==2.8.0
torchvision
torchaudio==2.8.0

accelerate==1.8.1
cn2an==0.5.22
cython==3.0.7
descript-audiotools==0.7.2
einops>=0.8.1
ffmpeg-python==0.2.0
g2p-en==2.1.0
jieba==0.42.1
json5==0.10.0
keras==2.9.0
librosa==0.10.2.post1
matplotlib==3.8.2
modelscope==1.27.0
munch==4.0.0
numba==0.58.1
numpy==1.26.2
omegaconf>=2.3.0
opencv-python==4.9.0.80
pandas==2.3.2
safetensors==0.5.2
sentencepiece>=0.2.1
tensorboard==2.9.1
textstat>=0.7.10
tokenizers==0.21.0
tqdm>=4.67.1
transformers==4.52.1
hf_xet
pydub
WeTextProcessing
gradio==6.11.0
"""

with open("/content/requirements_colab.txt", "w") as f:
    f.write(requirements)

print("Skrev /content/requirements_colab.txt")


Skrev /content/requirements_colab.txt


In [5]:
import os

os.environ["UV_SKIP_WHEEL_FILENAME_CHECK"] = "1"
os.environ["UV_LINK_MODE"] = "copy"

# Eget Python 3.10-miljo, isolert fra Colab sitt eget
!uv venv --python 3.10 /content/venv310

!uv pip install --python /content/venv310/bin/python -r /content/requirements_colab.txt --index-strategy unsafe-best-match

print("\nSteg 1 ferdig.")
!/content/venv310/bin/python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"


Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: /content/venv310
Activate with: source /content/venv310/bin/activate
Using Python 3.10.12 environment at: /content/venv310
Resolved 174 packages in 6.52s
Prepared 173 packages in 1m 51s
Installed 174 packages in 1m 34s
 + absl-py==2.5.0
 + accelerate==1.8.1
 + aiofiles==24.1.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.14.2
 + argbind==0.3.9
 + asttokens==3.0.2
 + audioread==3.1.0
 + brotli==1.2.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cn2an==0.5.22
 + contourpy==1.3.2
 + cryptography==50.0.1
 + cycler==0.12.1
 + cython==3.0.7
 + decorator==5.3.1
 + defusedxml==0.7.1
 + descript-audiotools==0.7.2
 + distance==0.1.3
 + docstring-parser==0.18.0
 + einops==0.8.2
 + exceptiongroup==1.3.1
 + executing==2.2.1
 + fastapi==0.141.1
 + ffmpeg-python==0.2.0
 + ffmpy==1.0.0
 + filelock==3.32.4
 + fire==0.7.1
 + 

## Steg 2 – Last ned modellene

Dette gjør det samme som `HF_model_downloader.py` i Patreon-pakka: henter hovedmodellen og
fire hjelpemodeller.

**Tar 5–10 minutter.** Til sammen noen gigabyte.

In [3]:
import os, shutil, glob

!pip install -q -U "huggingface_hub[hf_xet]"
from huggingface_hub import snapshot_download, hf_hub_download

APP  = "/content/Premium_IndexTTS2_SECourses"
CKPT = os.path.join(APP, "checkpoints")
CACHE = os.path.join(CKPT, "hf_cache")
os.makedirs(CKPT, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --- 1. Hovedmodellen -------------------------------------------------
print("Laster ned hovedmodellen ...")
tmp = "/content/_dl_tmp"
snapshot_download(
    repo_id="MonsterMMORPG/Wan_GGUF",
    allow_patterns=["Index_TTS2/*"],
    local_dir=tmp,
)

# Filene skal ligge rett i checkpoints/, ikke i en Index_TTS2-undermappe
kilde = os.path.join(tmp, "Index_TTS2")
for navn in os.listdir(kilde):
    maal = os.path.join(CKPT, navn)
    if os.path.exists(maal):
        shutil.rmtree(maal) if os.path.isdir(maal) else os.remove(maal)
    shutil.move(os.path.join(kilde, navn), maal)
shutil.rmtree(tmp, ignore_errors=True)

# --- 2. Hjelpemodellene ----------------------------------------------
hjelpemodeller = [
    ("facebook/w2v-bert-2.0", ["config.json", "model.safetensors", "preprocessor_config.json"]),
    ("amphion/MaskGCT", ["semantic_codec/model.safetensors"]),
    ("funasr/campplus", ["campplus_cn_common.bin"]),
    ("nvidia/bigvgan_v2_22khz_80band_256x", ["bigvgan_generator.pt", "config.json"]),
]

for repo, filer in hjelpemodeller:
    print(f"\nLaster ned {repo} ...")
    for fil in filer:
        hf_hub_download(repo_id=repo, filename=fil, repo_type="model", cache_dir=CACHE)

# --- 3. Kontroll ------------------------------------------------------
print("\n" + "=" * 55)
mangler = [f for f in ["bpe.model", "gpt.pth", "config.yaml", "s2mel.pth",
                       "wav2vec2bert_stats.pt"]
           if not os.path.exists(os.path.join(CKPT, f))]
if mangler:
    print("MANGLER filer:", mangler, "- kjor cella paa nytt")
else:
    print("Alle nodvendige modellfiler er paa plass.")
print("=" * 55)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 51.8 MB/s eta 0:00:00
Laster ned hovedmodellen ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]


Laster ned facebook/w2v-bert-2.0 ...


config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

...


Laster ned nvidia/bigvgan_v2_22khz_80band_256x ...


bigvgan_generator.pt: reconstructing file:   0%|          |  0.00B /  449MB            

bigvgan_generator.pt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]


Alle nodvendige modellfiler er paa plass.


## Steg 3 – Start appen

SECourses-appen har en innebygd delingsfunksjon (`--share`), så den lager selv en offentlig
lenke som slutter på `gradio.live`. Alle i gruppa kan åpne den samtidig.

Første oppstart tar **2–5 minutter**. Vent til dere ser `>>> ÅPEN LENKE`.

In [8]:
# ---------------------------------------------------------------
FP16 = False     # SETT TIL FALSE
# ---------------------------------------------------------------

import os, socket, subprocess, time

APP  = "/content/Premium_IndexTTS2_SECourses"
CKPT = os.path.join(APP, "checkpoints")
LOGG = "/content/webui.log"


def finn_lenke(tekst):
    """Plukker ut den siste gradio.live-adressen appen har skrevet ut."""
    treff = None
    for bit in tekst.split():
        if "gradio.live" in bit and bit.startswith("http"):
            treff = bit.strip().rstrip(".,")
    return treff


# Stopp en app fra en tidligere kjoring, saa porten er ledig
subprocess.run("pkill -f webui.py", shell=True)
time.sleep(2)

miljo = dict(os.environ)
miljo["PYTHONUNBUFFERED"] = "1"                              # VIKTIG: ellers blir loggen tom
miljo["PYTHONWARNINGS"]   = "ignore"
miljo["HF_HOME"]          = CKPT                             # som i Windows_Start_App.bat
miljo["HF_HUB_CACHE"]     = os.path.join(CKPT, "hf_cache")   # der Steg 2 la hjelpemodellene
miljo["GRADIO_ANALYTICS_ENABLED"] = "False"

args = ["/content/venv310/bin/python", "-u", "webui.py",
        "--model_dir", CKPT,
        "--host", "0.0.0.0", "--port", "7860",
        "--share"]
if FP16:
    args.append("--fp16")

app = subprocess.Popen(args, cwd=APP, env=miljo,
                       stdout=open(LOGG, "w"), stderr=subprocess.STDOUT)

print("Starter appen. Alt den skriver ut vises under:")
print("-" * 68)

leser = open(LOGG)
alt, url = "", None
start = time.time()

while time.time() - start < 900:                   # maks 15 minutter
    ny = leser.read()
    if ny:
        print(ny, end="")
        alt += ny
    url = finn_lenke(alt)
    if url:
        break
    if app.poll() is not None:
        time.sleep(1)
        print(leser.read(), end="")
        break
    time.sleep(1)

print()
print("=" * 68)
if url:
    print()
    print("   BRUK DENNE LENKEN - kopier den og apne i en ny fane:")
    print()
    print("      " + url)
    print()
    print("   Ikke bruk 127.0.0.1-adressen. Den peker paa Googles maskin,")
    print("   ikke paa din egen.")
    print()
else:
    print("   Fant ingen delingslenke. Se utskriften over etter feilmelding.")
print("=" * 68)
print()
print("Kjor cella under for a folge loggen mens dere jobber i appen.")


Starter appen. Alt den skriver ut vises under:
--------------------------------------------------------------------
2026-08-28 09:46:59,861 WETEXT INFO found existing fst: /content/Premium_IndexTTS2_SECourses/indextts/utils/tagger_cache/zh_tn_tagger.fst
2026-08-28 09:46:59,861 WETEXT INFO                     /content/Premium_IndexTTS2_SECourses/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
2026-08-28 09:46:59,861 WETEXT INFO skip building fst for zh_normalizer ...
2026-08-28 09:47:00,111 WETEXT INFO found existing fst: /content/venv310/lib/python3.10/site-packages/tn/en_tn_tagger.fst
2026-08-28 09:47:00,111 WETEXT INFO                     /content/venv310/lib/python3.10/site-packages/tn/en_tn_verbalizer.fst
2026-08-28 09:47:00,111 WETEXT INFO skip building fst for en_normalizer ...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7ac46ebcd25ffcad26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio 

### Følg loggen mens dere jobber

Denne cella viser alt appen skriver ut, fortløpende — det samme som ville stått i et
terminalvindu på en vanlig PC. **La den kjøre i bakgrunnen mens dere bruker appen i den andre
fana.** Går en generering galt og appen bare sier «FEIL», er det her den ekte feilmeldingen står.

Cella kjører til dere stopper den. Trykk stopp-knappen når dere skal kjøre en annen celle.

In [ ]:
import time

LOGG = "/content/webui.log"

print("Logg fra appen. Trykk stopp-knappen for a kjore en annen celle.")
print("-" * 68)

with open(LOGG) as f:
    for linje in f.readlines()[-40:]:      # vis det som allerede har skjedd
        print(linje, end="")
    try:
        while True:                        # og folg med videre
            ny = f.read()
            if ny:
                print(ny, end="")
            else:
                time.sleep(1)
    except KeyboardInterrupt:
        print()
        print("-" * 68)
        print("Sluttet a folge loggen. Appen kjorer videre.")


Logg fra appen. Trykk stopp-knappen for a kjore en annen celle.
--------------------------------------------------------------------
2026-08-28 09:46:59,861 WETEXT INFO found existing fst: /content/Premium_IndexTTS2_SECourses/indextts/utils/tagger_cache/zh_tn_tagger.fst
2026-08-28 09:46:59,861 WETEXT INFO                     /content/Premium_IndexTTS2_SECourses/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
2026-08-28 09:46:59,861 WETEXT INFO skip building fst for zh_normalizer ...
2026-08-28 09:47:00,111 WETEXT INFO found existing fst: /content/venv310/lib/python3.10/site-packages/tn/en_tn_tagger.fst
2026-08-28 09:47:00,111 WETEXT INFO                     /content/venv310/lib/python3.10/site-packages/tn/en_tn_verbalizer.fst
2026-08-28 09:47:00,111 WETEXT INFO skip building fst for en_normalizer ...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7ac46ebcd25ffcad26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgra

## Steg 4 – Slik bruker dere grensesnittet

Dette er SECourses-versjonen, så den ser annerledes ut enn standard IndexTTS.

**Referansestemme:** last opp lydfila, eller ta opp rett fra mikrofonen i nettleseren
(anbefalt lengde 15 sekunder). Appen godtar også videofiler og trekker ut lyden selv.

**Emotion control** har fire valg:

| Valg | Hva det gjør |
|---|---|
| *Same as speaker voice* | Følelsen hentes fra referanseopptaket |
| *Use emotion reference audio* | Et **eget** klipp styrer bare følelsen |
| *Use emotion vector control* | Åtte skyveknapper: glad, sint, trist, redd, avsky, tungsindig, overrasket, rolig |
| *Use emotion text description* | Beskriv følelsen med ord på engelsk, f.eks. `whispering, secretive` |

**Nyttige knapper i denne versjonen:**

- **Save / Load preset** — lagre innstillingene deres og hent dem tilbake. Bruk dette. Da kan
  dere sammenligne to oppsett uten å skrive alt om igjen.
- **Cancel** — stopp en generering som har låst seg
- **Max tokens per segment** — lange tekster deles opp automatisk
- Ferdige filer nummereres og legges i `outputs/`

Ett råd: **endre én ting av gangen**, og lagre en preset når dere finner noe som funker.

Går en generering galt, se i loggcella lenger opp. Der står den ekte feilmeldingen.

## Steg 5 – Last ned lydfilene

Kjør denne **før dere lukker Colab.** Alt forsvinner ellers.

Følger dere loggen i cella over, må dere stoppe den først.

In [ ]:
import shutil, os
from google.colab import files

UT = "/content/Premium_IndexTTS2_SECourses/outputs"
os.makedirs(UT, exist_ok=True)

print("Filer i outputs/:")
!ls -R "$UT" | head -50

shutil.make_archive("/content/lydfiler", "zip", UT)
files.download("/content/lydfiler.zip")
